## 0. Конфиг

In [ ]:
from pathlib import Path

CFG = {
    "graph_dir": "/home/nikita/DL/recsys/favegraph/favegraph_frac0.02",
    "ckpt_dir": "outputs/favegraph_ckpts",
    "seed": 42,
    "embedding_dim": 128,
    "batch_size": 4096,
    "epochs": 30,
    "lr": 0.05,
    "weight_decay": 1.0e-05,
    "negative_samples": 64,
    "grad_clip_norm": 1.0,
    "mixed_precision": True,
    "num_workers": 0,
    "neg_power": 0.75,
    "corrupt": "both",
    "recall_ks": [10, 20, 50],
    "early_stop_patience": 5,
}
Path(CFG["ckpt_dir"]).mkdir(parents=True, exist_ok=True)
CFG

{'graph_dir': '/home/nikita/DL/recsys/favegraph/favegraph_frac0.02',
 'ckpt_dir': 'outputs/favegraph_ckpts',
 'seed': 42,
 'embedding_dim': 128,
 'batch_size': 4096,
 'epochs': 30,
 'lr': 0.05,
 'weight_decay': 1e-05,
 'negative_samples': 64,
 'grad_clip_norm': 1.0,
 'mixed_precision': True,
 'num_workers': 0,
 'neg_power': 0.75,
 'corrupt': 'both',
 'recall_ks': [10, 20, 50],
 'early_stop_patience': 5}

## 1. Загрузка готового графа

In [ ]:
import numpy as np
import pandas as pd

G = Path(CFG["graph_dir"])
tri = pd.read_parquet(G / "triples.parquet")  # lhs, rel, rhs, split
ent = pd.read_parquet(G / "entities.parquet")  # entity_id, entity_type, orig_id
rel_tab = pd.read_parquet(G / "relations.parquet")

n_users = int((ent["entity_type"] == "user").sum())
n_ent = len(ent)
n_rel = len(rel_tab)
REL_FAVE = 0

train_df = tri[tri["split"] == "train"][["lhs", "rel", "rhs"]].reset_index(drop=True)
val_df   = tri[tri["split"] == "val"][["lhs", "rel", "rhs"]].reset_index(drop=True)
test_df  = tri[tri["split"] == "test"][["lhs", "rel", "rhs"]].reset_index(drop=True)

print(f"entities: {n_ent:,} (users={n_users:,}, tweets={n_ent-n_users:,})")
print(f"edges: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")


entities: 2,065,623 (users=262,614, tweets=1,803,009)
edges: train=5,263,114 val=99,173 test=74,247


## 2. Модель

In [3]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["seed"])
print("device:", DEVICE)

class TransE(nn.Module):
    def __init__(self, n_entities, n_relations, dim):
        super().__init__()
        self.ent = nn.Embedding(n_entities, dim)
        self.rel = nn.Embedding(n_relations, dim)
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)

    def score(self, h, r, t):
        return ((self.ent(h) + self.rel(r)) * self.ent(t)).sum(-1)

    def tail_scores(self, h, r, cand_t):
        q = (self.ent(h) + self.rel(r))[:, None, :]
        return (q * self.ent(cand_t)).sum(-1)

    def head_scores(self, cand_h, r, t):
        q = (self.ent(t) - self.rel(r))[:, None, :]
        return (q * self.ent(cand_h)).sum(-1)

device: cuda


## 3. Типизированный негатив-сэмплер (user / tweet)

In [ ]:
class TypedSampler:
    def __init__(self, edges_df, n_users, n_ent, strategy, power):
        self.strategy = strategy
        self.user_ids = torch.arange(0, n_users)
        self.item_ids = torch.arange(n_users, n_ent)  # tweets
        self.n_users, self.n_ent = n_users, n_ent
        if strategy == "frequency":
            cnt = np.ones(n_ent)
            vc = pd.concat([edges_df["lhs"], edges_df["rhs"]]).value_counts()
            cnt[vc.index.to_numpy()] = vc.to_numpy()
            cnt = cnt ** power
            self.pu = torch.tensor(cnt[:n_users] / cnt[:n_users].sum(), dtype=torch.float)
            self.pi = torch.tensor(cnt[n_users:] / cnt[n_users:].sum(), dtype=torch.float)
            self.logq = torch.tensor(np.log(cnt / cnt.sum() + 1e-12), dtype=torch.float)
        else:
            self.logq = torch.full((n_ent,), -np.log(n_ent), dtype=torch.float)

    def _sample(self, is_user, n, device, gen):
        if self.strategy == "uniform":
            base = self.user_ids if is_user else self.item_ids
            return base[torch.randint(0, len(base), (n,), generator=gen)].to(device)
        p = self.pu if is_user else self.pi
        base = self.user_ids if is_user else self.item_ids
        return base[torch.multinomial(p, n, replacement=True, generator=gen)].to(device)

    def sample_like(self, ref_ids, k, device, gen=None):
        is_user = ref_ids < self.n_users
        out = torch.empty((ref_ids.size(0), k), dtype=torch.long, device=device)
        if is_user.any():
            out[is_user] = self._sample(True, int(is_user.sum()) * k, device, gen).view(-1, k)
        if (~is_user).any():
            out[~is_user] = self._sample(False, int((~is_user).sum()) * k, device, gen).view(-1, k)
        return out

    def log_q(self, ids, device):
        return self.logq.to(device)[ids]

## 4. Лоссы: NS , sampled-softmax, sampled-softmax + logQ

In [5]:
import torch.nn.functional as F

def ns_loss(pos, neg):
    return -(F.logsigmoid(pos) + F.logsigmoid(-neg).mean(1)).mean()

def sampled_softmax_loss(pos, neg, log_q=None):
    logits = torch.cat([pos[:, None], neg], dim=1)
    if log_q is not None:
        logits = logits - log_q.clamp(min=-20.0)
    target = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, target)

def compute_loss(model, h, r, t, sampler, neg_k, loss_name, corrupt, use_logq, gen=None):
    scale = model.ent.weight.size(1) ** 0.5
    pos = model.score(h, r, t)
    negs, neg_ids = [], []
    if corrupt in ("tail", "both"):
        k = neg_k // 2 if corrupt == "both" else neg_k
        nt = sampler.sample_like(t, k, h.device, gen)
        negs.append(model.tail_scores(h, r, nt)); neg_ids.append(nt)
    if corrupt in ("head", "both"):
        k = neg_k - neg_k // 2 if corrupt == "both" else neg_k
        nh = sampler.sample_like(h, k, h.device, gen)
        negs.append(model.head_scores(nh, r, t)); neg_ids.append(nh)
    neg = torch.cat(negs, 1)
    if loss_name == "ns":
        return ns_loss(pos, neg)
    log_q = None
    if use_logq:
        cand = torch.cat(neg_ids, 1)
        pos_t = t if corrupt != "head" else h
        log_q = torch.cat([sampler.log_q(pos_t[:, None], h.device), sampler.log_q(cand, h.device)], 1)
    return sampled_softmax_loss(pos / scale, neg / scale, log_q)

## 5. Обучение

In [6]:
import faiss
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

def make_loader(df, shuffle):
    ds = TensorDataset(torch.tensor(df["lhs"].values), torch.tensor(df["rel"].values),
                       torch.tensor(df["rhs"].values))
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=(DEVICE.type == "cuda"), drop_last=shuffle)

def build_gt_known(gt_df, *known_dfs):
    gt = {}
    for u, t in zip(gt_df["lhs"], gt_df["rhs"]):
        gt.setdefault(int(u), set()).add(int(t))
    known = {}
    for kdf in known_dfs:
        for u, t in zip(kdf["lhs"], kdf["rhs"]):
            known.setdefault(int(u), set()).add(int(t))
    return gt, known

_VAL_GT, _VAL_KNOWN = build_gt_known(val_df, train_df)

@torch.no_grad()
def quick_recall(model, k=10, max_q=5000):
    model.eval()
    e = model.ent.weight.detach().cpu().numpy().astype("float32")
    relv = model.rel.weight.detach().cpu().numpy()[REL_FAVE].astype("float32")
    tweet = e[n_users:] / np.linalg.norm(e[n_users:], axis=1, keepdims=True).clip(min=1e-12)
    index = faiss.IndexFlatIP(e.shape[1]); index.add(tweet)
    qs = list(_VAL_GT.keys())[:max_q]
    Q = e[qs] + relv
    Q = Q / np.linalg.norm(Q, axis=1, keepdims=True).clip(min=1e-12)
    _, I = index.search(Q, k + 60)
    hits = []
    for u, row in zip(qs, I):
        seen = _VAL_KNOWN.get(u, set())
        cand = [int(x) + n_users for x in row if (int(x) + n_users) not in seen][:k]
        hits.append(1.0 if set(cand) & _VAL_GT[u] else 0.0)
    return float(np.mean(hits)) if hits else 0.0

def train(loss_name="ns", use_logq=False, neg_strategy="frequency"):
    train_loader = make_loader(train_df, True)
    sampler = TypedSampler(train_df, n_users, n_ent, neg_strategy, CFG["neg_power"])
    model = TransE(n_ent, n_rel, CFG["embedding_dim"]).to(DEVICE)
    opt = torch.optim.Adagrad(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    amp = CFG["mixed_precision"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    tag = f"{loss_name}" + ("__logq" if use_logq else "") + f"__{neg_strategy}"
    best_r, curve, bad = -1.0, [], 0
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); run = 0.0
        pbar = tqdm(train_loader, desc=f"{tag} ep{ep}")
        for h, r, t in pbar:
            h, r, t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = compute_loss(model, h, r, t, sampler, CFG["negative_samples"],
                                    loss_name, CFG["corrupt"], use_logq)
            scaler.scale(loss).backward()
            if CFG["grad_clip_norm"] > 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
            scaler.step(opt); scaler.update()
            run += loss.item(); pbar.set_postfix(loss=f"{run/(pbar.n+1):.4f}")
        rec = quick_recall(model, k=CFG["recall_ks"][0])
        curve.append(rec); print(f"  ep{ep} train_loss={run/len(train_loader):.4f}  val_Recall@10={rec:.4f}")
        if rec > best_r:
            best_r = rec; bad = 0
            torch.save({"state_dict": model.state_dict(), "n_ent": n_ent, "n_rel": n_rel,
                        "dim": CFG["embedding_dim"], "tag": tag}, Path(CFG["ckpt_dir"]) / f"{tag}.pt")
        else:
            bad += 1
            if bad >= CFG["early_stop_patience"]:
                print(f"  early stop ep{ep}"); break
    print(f"  best val_Recall@10={best_r:.4f}")
    return tag, curve

## 6. Candidate generation (user→tweet) + бейзлайны

In [ ]:
from sklearn.linear_model import LogisticRegression

def load_model_ckpt(tag):
    ck = torch.load(Path(CFG["ckpt_dir"]) / f"{tag}.pt", map_location="cpu")
    m = TransE(ck["n_ent"], ck["n_rel"], ck["dim"]); m.load_state_dict(ck["state_dict"])
    return m

def metrics(ranked, gt, ks):
    rec = {k: [] for k in ks}; mrr = []
    for u, pos in gt.items():
        if u not in ranked: continue
        c = ranked[u]
        for k in ks:
            rec[k].append(len(set(c[:k]) & pos) / len(pos))
        rr = 0.0
        for i, e in enumerate(c, 1):
            if e in pos: rr = 1 / i; break
        mrr.append(rr)
    out = {"MRR": np.mean(mrr) if mrr else 0.0}
    for k in ks: out[f"R@{k}"] = np.mean(rec[k]) if rec[k] else 0.0
    return out

def candidate_generation(tag, ks):
    m = load_model_ckpt(tag)
    e = m.ent.weight.detach().numpy().astype("float32")
    relv = m.rel.weight.detach().numpy()[REL_FAVE].astype("float32")
    gt, known = build_gt_known(test_df, train_df, val_df)
    max_k = max(ks)

    tweet = e[n_users:] / np.linalg.norm(e[n_users:], axis=1, keepdims=True).clip(min=1e-12)
    index = faiss.IndexFlatIP(e.shape[1]); index.add(tweet)

    qs = list(gt.keys())
    Q = e[qs] + relv
    Q = Q / np.linalg.norm(Q, axis=1, keepdims=True).clip(min=1e-12)
    _, I = index.search(Q, max_k + 200)
    model_rank = {}
    for u, row in zip(qs, I):
        seen = known.get(u, set())
        model_rank[u] = [int(x) + n_users for x in row if (int(x) + n_users) not in seen][:max_k]

    # бейзлайны
    pop = train_df["rhs"].value_counts()
    popular = [int(x) for x in pop.index[:max_k * 3]]
    rng = np.random.default_rng(0)
    randl, popl = {}, {}
    for u in qs:
        seen = known.get(u, set())
        randl[u] = [int(x) for x in rng.integers(n_users, n_ent, max_k)]
        popl[u] = [x for x in popular if x not in seen][:max_k]

    return {"TwHIN": metrics(model_rank, gt, ks),
            "popular": metrics(popl, gt, ks),
            "random": metrics(randl, gt, ks)}

## 7. Главный эксп: лосс и negative sampling

In [ ]:
runs = [
    ("ns", False, "uniform"),
    ("ns", False, "frequency"),
    ("sampled_softmax", False, "frequency"),
    ("sampled_softmax", True,  "frequency"),  # + logQ-коррекция
]

rows = []
for loss_name, logq, negs in runs:
    tag, _ = train(loss_name=loss_name, use_logq=logq, neg_strategy=negs)
    r = candidate_generation(tag, CFG["recall_ks"])["TwHIN"]
    rows.append({"loss": loss_name, "logq": logq, "neg": negs,
                 **{f"R@{k}": round(r[f'R@{k}'] * 100, 3) for k in CFG["recall_ks"]},
                 "MRR": round(r["MRR"], 4)})

# бейзлайны для нижней границы
base = candidate_generation(tag, CFG["recall_ks"])
for name in ("popular", "random"):
    b = base[name]
    rows.append({"loss": name, "logq": "", "neg": "",
                 **{f"R@{k}": round(b[f'R@{k}'] * 100, 3) for k in CFG["recall_ks"]},
                 "MRR": round(b["MRR"], 4)})

pd.DataFrame(rows)

ns__uniform ep1:   0%|          | 0/1284 [00:00<?, ?it/s]

  ep1 train_loss=0.9458  val_Recall@10=0.0058


ns__uniform ep2:   0%|          | 0/1284 [00:00<?, ?it/s]

  ep2 train_loss=0.8820  val_Recall@10=0.0032


ns__uniform ep3:   0%|          | 0/1284 [00:00<?, ?it/s]

...

sampled_softmax__logq__frequency ep6:   0%|          | 0/1284 [00:00<?, ?it/s]

  ep6 train_loss=3.5871  val_Recall@10=0.0000


sampled_softmax__logq__frequency ep7:   0%|          | 0/1284 [00:00<?, ?it/s]

  ep7 train_loss=3.5803  val_Recall@10=0.0000
  early stop ep7
  best val_Recall@10=0.0002


,loss,logq,neg,R@10,R@20,R@50,MRR
0,ns,False,uniform,0.042,0.101,0.309,0.0006
1,ns,False,frequency,0.000,0.002,0.031,0.0000
2,sampled_softmax,False,frequency,0.000,0.001,0.002,0.0000
3,sampled_softmax,True,frequency,0.000,0.000,0.000,0.0000
4,popular,,,0.019,0.038,0.104,0.0004
5,random,,,0.000,0.000,0.003,0.0000
